## Demo

Physics-informed AI-based surrogate model for efficient What-If analysis.

This Python implementation is based on the [EPyT-Control](https://github.com/WaterFutures/EPyT-Control) package.
You can find more information about the implemented surrogate models in the documentation of EPyT-Control -- e.g., https://epyt-control.readthedocs.io/en/stable/tut.hyd_surrogate.html and https://epyt-control.readthedocs.io/en/stable/tut.qual_surrogate.html

**DISCLAIMER:** This example/demo is not meant to be realistic! The aim is to demonstrate how to implement this specific concept in Python and how to use existing Python packages.

In [ ]:
%pip install torch==2.10.0 --index-url https://download.pytorch.org/whl/cpu
%pip install torch-geometric==2.6.1
%pip install torch-scatter -f https://data.pyg.org/whl/torch-2.10.0+cpu.html
%pip install epyt-control[hydsurrogate]

In [ ]:
!git clone https://github.com/andreArtelt/IAHR26ExplainableML4WDS.git

In [ ]:
import random
random.seed(4242)
import numpy as np
np.random.seed(422)
import matplotlib.pyplot as plt
import torch
import sys
sys.path.insert(0,'/content/IAHR26ExplainableML4WDS/case-study-2')

from epyt_flow.utils import plot_timeseries_data
from epyt_control.models import PIGNNModel, device
from generate_data import generate_data, pressure_sensor_locations

Create and load a [pre-trained hydraulic surrogate model](https://epyt-control.readthedocs.io/en/stable/tut.hyd_surrogate.html) for the [Hanoi](https://waterfutures.github.io/WaterBenchmarkHub/benchmarks/network-Hanoi.html) benchmark network:

<img src="https://waterfutures.github.io/WaterBenchmarkHub/static/benchmarks/network-hanoi/hanoi_plot.png" style="width:600px;height:400px;">
<br>
<img src="pignn.png">

See https://ieeexplore.ieee.org/document/11229349 for more details on the physics-informed Graph Neural Network.

In [ ]:
model = PIGNNModel.from_network("hanoi", load_pretrained_model=True)

Create scenarios (21-days long) based on [Hanoi](https://waterfutures.github.io/WaterBenchmarkHub/benchmarks/network-Hanoi.html) from [LeakDB](https://waterfutures.github.io/WaterBenchmarkHub/benchmarks/KIOS-LeakDB.html) -- include random sensor faults (i.e., adding Gaussian noise).

In [ ]:
# Generate data
X_train, y_train, hyd_sim_data_plain_train = generate_data(scenarios_id=["1", "2", "3"], apply_sensor_faults=False, apply_leakages=False)
X_test, y_test, hyd_sim_data_plain_test = generate_data(scenarios_id=["10"], apply_sensor_faults=False, apply_leakages=True)
print(X_train.shape)
print(X_test.shape)

Run the hydraulic surrogate model -- note that we only need heads at the reservoir and demands at all nodes. In practice, heads at the reservoirs might be known due to sensor readings, and demands could be estimated and predicted as part of a digital twin.

In [ ]:
heads, reservoir_idx, demands, flows = hyd_sim_data_plain_train[0]  # First training scenario without any faults

# Run surrogate model
heads_pred, demands_pred, flows_pred = model.predict_as_numpy(reservoir_heads=heads[:, reservoir_idx],
                                                              demands=demands)

The surrogate model does a good job most of the time, but it is not perfect!

In [ ]:
# Compare predictions to ground truth
plot_timeseries_data(np.abs(heads_pred - heads).T,
                     x_axis_label="Time",
                     y_axis_label="Absolute pressure head [m] error")
plot_timeseries_data(np.abs(flows_pred - flows).T,
                     x_axis_label="Time",
                     y_axis_label="Absolute flow rate [m^3/s] error")

In [ ]:
plot_timeseries_data(heads.T,
                     x_axis_label="Time",
                     y_axis_label="Pressure head [m]")

## Compare surrogate model to sensor readings

Apply the hydraulic surrogate model to a scenario with a fault:

In [ ]:
heads, reservoir_idx, demands, flows = hyd_sim_data_plain_test[0]
pressure_obs = X_test    # Observed pressure readings from a (faulty) sensor

hyd_pred = model.predict_as_scada_data(reservoir_heads=heads[:, reservoir_idx],
                                       demands=demands)
pressure_pred = hyd_pred.get_data_pressures(pressure_sensor_locations)

Compare predictions to ground truth (i.e., sensor readings) -- note that the observed sensor readings are affected by the fault fault!

In [ ]:
plot_timeseries_data(np.abs(pressure_pred - pressure_obs).T,
                     x_axis_label="Time",
                     y_axis_label="Absolute pressure [m] error",
                     labels=[f"Node {i}" for i in pressure_sensor_locations])

## Gradient-based analysis

One of the big advantages of a neural surrogate model is its differentiability -- i.e., we can compute gradients w.r.t. different quantities (e.g., demands, pipe parameters, etc.)

In the following, we want to investigate the sensivitive of pressure heads to consumer/node demands.

In [ ]:
heads_obs, reservoir_idx, demands, flows = hyd_sim_data_plain_test[0]  # Scenario with a fault
heads_pred, _, _ = model.predict_as_numpy(reservoir_heads=heads[:, reservoir_idx],
                                          demands=demands)

We have to pick a time point, for which we want to compute the sensitivity:

In [ ]:
#t = 100
t = 500     # We know that there is the fault active at this point in time!

In [ ]:
# Compare predicted pressure heads to observed heads (i.e., from the faulty sensor)
headsobs = torch.tensor(heads_obs[t, :], dtype=torch.float32).to(device)
loss = lambda heads_pred: (heads_pred - headsobs).abs().sum().view(-1, 1)

In [ ]:
g = model.compute_gradients(reservoir_heads=heads[t, reservoir_idx].reshape(1, -1),
                            demands=demands[t, :].reshape(1, -1),
                            gradient_output="heads",   # We are targeting the predicted pressure heads, not the flow rates!
                            gradient_input="demands",  # We compute gradients w.r.t. the demands
                            output_func=loss)

print(g["grads"].shape)  # Get gradients' shape -- note that they are returned as a torch.Tensor

# Convert gradient to NumPy array
demand_grad = g["grads"].detach().numpy().flatten()

In [ ]:
demand_grad

In [ ]:
plt.figure()
plt.bar([f"Node {n_id+1}" for n_id in range(demand_grad.shape[0])], demand_grad / np.sum(demand_grad))
plt.xlabel("Node")
plt.xticks(rotation=90)
plt.ylabel("Normalized demand gradient")
plt.show()